# Agentic RAG — From Scratch to Agent
*by [Jusnaini](https://www.linkedin.com/in/jusnainimuslimin/)*

This notebook documents my solution to the **Module 1 homework of [LLM Zoomcamp](https://github.com/DataTalksClub/llm-zoomcamp/tree/main/cohorts/2026/01-agentic-rag)** by DataTalksClub. I'm sharing it as a walkthrough of how I approached each problem, in case it's useful to others going through the same material.

## What's covered

Starting from a raw [GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp/tree/main) of the course lesson pages, I built a full RAG pipeline step by step — 
- fetching and indexing documents, 
- writing the search-prompt-LLM loop from scratch, 
- experimenting with chunking to reduce prompt size, 
- and finally turning the whole thing into an agent that decides on its own when and what to search.

The six homework questions are answered along the way, but the focus is on understanding *why* each piece works the way it does, not just getting the right answer.

## Stack I used

| Component | Tool |
|---|---|
| Document source | `gitsource` (GitHub repo reader) |
| Search index | `minsearch` |
| LLM providers | `Groq` / `OpenRouter`|
| Agent framework | `toyaikit` |

> API keys are loaded from a local `config.py` file. Swap in your own client setup if you'd like to run the cells yourself.

## Client API Setup

This notebook interfaces with ***[Groq](https://groq.com)*** ***(or [OpenRouter](https://openrouter.ai/))*** using a centralized configuration wrapper (`config.py`) that manages environment keys and simplifies client initialization.

Running the cell below dynamically resolves the project root directory(the folder containing `config.py`), appends it to `sys.path` for local imports, and loads the pre-configured client constructors.


In [1]:
import sys
from pathlib import Path

# Identify current workspace directory
current_dir = Path(__file__).resolve().parent if '__file__' in locals() else Path.cwd()

# Traverse upward to locate project root containing config.py
root_dir = current_dir
while root_dir.parent != root_dir:
    if (root_dir / "config.py").exists():
        break
    root_dir = root_dir.parent

# Expose project root to Python search path for local imports
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))

# Import openai client api calls
from config import get_groq_client, get_openrouter_client

text = """
# Quick Start Usage:

llm_client = get_groq_client()
OR
llm_client = get_openrouter_client()
"""
print("✅ Notebook environment setup complete!")
print(text)

✅ Notebook environment setup complete!

# Quick Start Usage:

llm_client = get_groq_client()
OR
llm_client = get_openrouter_client()



### API test

Before running the full pipeline, confirm both API providers are reachable and returning sensible output. Each helper prints the endpoint, model, question, response, and total token count.

In [2]:
# test function
question = "How many r's are in the word 'strawberry'?"

def test_groq(
    client=get_groq_client(),
    question=question,
    model="openai/gpt-oss-120b"):
    
    response = client.responses.create(
        input=question,
        model=model,
    )
    output = response.output_text
    total_tokens = response.usage.total_tokens 
    print(f"\
        Endpoint: Groq\n\
        Model   : {model} \n\
        Question: {question} \n\
        {'--'*30}\n\
        Response: {output}\n\
        #Tokens : {total_tokens}\n\n")

def test_openrouter(
    client=get_openrouter_client(),
    question=question,
    model="openai/gpt-oss-120b:free"):
    
    response = client.chat.completions.create(
        messages=[{"role": "user", "content": question}], 
        model=model)
    output = response.choices[0].message.content
    total_tokens = response.usage.total_tokens 
    print(f"\
        Endpoint: OpenRouter\n\
        Model   : {model} \n\
        Question: {question} \n\
        {'--'*30}\n\
        Response: {output}\n\
        #Tokens : {total_tokens}\n\n")


In [3]:
# run test()
test_groq()
test_openrouter()

        Endpoint: Groq
        Model   : openai/gpt-oss-120b 
        Question: How many r's are in the word 'strawberry'? 
        ------------------------------------------------------------
        Response: There are **3** r’s in the word “strawberry.”
        #Tokens : 235


        Endpoint: OpenRouter
        Model   : openai/gpt-oss-120b:free 
        Question: How many r's are in the word 'strawberry'? 
        ------------------------------------------------------------
        Response: The word **“strawberry”** contains **3** r’s.
        #Tokens : 128




## Prepare the Knowledge Base

First, we pull the course lesson pages straight from GitHub using `gitsource`.

**Pinning the commit** (`8c1834d`) ensures every run works with the exact same data, regardless of future repository changes. The `filename_filter` keeps only files inside a module's `lessons/` subfolder, skipping top-level READMEs and other markdown.

In [4]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [5]:
print(type(files))
files[0]

<class 'list'>


RawRepositoryFile(filename='01-agentic-rag/lessons/01-intro.md', content='# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are 

### **Q1 — How many lesson pages are in the dataset?**

Count the raw `RawRepositoryFile` objects returned by the reader. Each object maps to one lesson markdown file.

In [6]:
# Q1: Count lesson pages downloaded from the repo
len(files)  # expected: 72

72

In [7]:
documents = []
for file in files:
    doc=file.parse()
    documents.append(doc)

In [ ]:
documents[:5]

[{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a sim

## Build a Search Index (minsearch)

Index the parsed documents with **minsearch** — a lightweight in-memory search engine.

- `content` is declared as a text field (full-text search)
- `filename` is declared as a keyword field (exact-match filter)

A quick test search on `'orchestration'` confirms the index is working before we run the homework query.

In [9]:
from minsearch import Index

index = Index(
    text_fields =["content"],
    keyword_fields=["filename"]
)
index.fit(documents)

In [10]:
index.search('orchestration')

[{'content': "# AI Orchestration\n\nVideo: [Using AI in Workflows](https://youtu.be/l9w6oeUtrpk)\n\nThis module introduces AI-powered workflow orchestration using Kestra. You'll learn how AI can accelerate workflow development and enable autonomous task automation through three techniques: AI Copilot for generating flows, RAG for grounding responses in real data, and AI Agents for autonomous task execution.\n\n## Learning Objectives\n\nBy the end of this module, you will:\n\n- Understand the importance of context engineering in AI applications\n- Use AI Copilot to build Kestra flows faster and more accurately\n- Implement Retrieval Augmented Generation (RAG) to ground AI responses in real data\n- Build autonomous AI agents that can make decisions and use tools dynamically\n- Design multi-agent systems where specialized agents collaborate to solve complex tasks\n- Apply best practices for using AI in production data workflows\n\n## Prerequisites\n\n- Kestra running locally (see [Setting

### **Q2 — What is the `filename` of the top search result?**

Run the exact homework query against the index and inspect the first hit.

In [11]:
# Q2: Run the homework query and check the top-ranked filename
results = index.search("How does the agentic loop keep calling the model until it stops?")
results[0]['filename']

'01-agentic-rag/lessons/14-agentic-loop.md'

## Build a RAG Pipeline (plain Python)

Rather than using `RAGBase` directly, let first build the pipeline from scratch so each component is visible:

1. `search` — retrieves relevant documents from the index  
2. `build_context` — formats the documents into a readable context block  
3. `build_prompt` — combines the user question with the context  
4. LLM call — sends the prompt to the model and reads the response

This mirrors the three-step architecture from the course: *search → prompt → LLM*.

In [12]:
# !wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py

In [13]:
documents[:2]

[{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a sim

#### Step 1 — Search and build context

Retrieve the top-5 matching documents and format them as a `filename: …` / `content: …` block.

In [14]:
def search(query, num_results=5,index=index):
    # boost_dict = {'question': 3.0, 'section': 0.5}
    # filter_dict = {'course': self.course}

    return index.search(
        query,
        num_results=num_results,
        # boost_dict=boost_dict,
        # filter_dict=filter_dict
    )

In [15]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append('filename: ' + doc['filename'])
        lines.append('content: ' + doc['content'])
        lines.append('')

    return '\n'.join(lines).strip()

In [16]:
search_results = search(query="How does the agentic loop keep calling the model until it stops?")
build_context(search_results)

'filename: 01-agentic-rag/lessons/14-agentic-loop.md\ncontent: # The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n-

#### Step 2 — Build the prompt

Wrap the question and the retrieved context in a structured prompt template.

In [17]:
prompt_template = '''
QUESTION: {question}

CONTEXT:
{context}
'''.strip()

def build_prompt(query, search_results):
    context = build_context(search_results)
    return prompt_template.format(question=query, context=context)

In [18]:
query="How does the agentic loop keep calling the model until it stops?"
prompt = build_prompt(query, search_results)

In [19]:
prompt

'QUESTION: How does the agentic loop keep calling the model until it stops?\n\nCONTEXT:\nfilename: 01-agentic-rag/lessons/14-agentic-loop.md\ncontent: # The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as th

#### Step 3 — Call the LLM

Send the prompt to the model via the Groq endpoint and collect the full response object (we need `response.usage` for Q3).

In [20]:
llm_client = get_groq_client()
model = "openai/gpt-oss-120b" 

In [21]:
instructions = '''
Your task is to answer question from the lesson content based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the lessons,
respond with "I don't know."
'''

input_messages = [
    {'role': 'developer', 'content': instructions},
    {'role': 'user', 'content': prompt}
]

response = llm_client.responses.create(
    model=model,
    input=input_messages
)

#### Step 4 — Inspect the answer

In [22]:
print(f"🤖 Response: {response.output_text}")

🤖 Response: The agentic loop is just a **while‑True** that repeatedly sends the current conversation to the model, runs any tool calls that the model requests, and then checks whether the model asked for another tool.

```python
it = 1
while True:
    print(f"iteration #{it}…")
    has_function_calls = False

    # 1️⃣ Call the model with the full message history
    response = openai_client.responses.create(
        model=model,
        input=messages,
        tools=[search_tool],
    )

    # 2️⃣ Append everything the model returned
    messages.extend(response.output)

    # 3️⃣ Walk through the model’s output
    for item in response.output:
        if item.type == "function_call":
            # run the requested tool
            call_output = make_call(item)
            messages.append(call_output)      # add the tool result
            has_function_calls = True         # remember we needed a tool
        elif item.type == "message":
            # just a normal assistant message –

### **Q3 — How many input tokens did we send for this RAG request?**

Read `usage.input_tokens` from the response object.  
*(We count input tokens rather than cost because the prompt size is the same regardless of model or provider.)*

In [23]:
# Q3: Read input (prompt) token count from the response usage object
usage = response.usage
print("-" * 30)
print(f"📥 Input (Prompt) Tokens: {usage.input_tokens}")
print(f"📤 Output (Completion) Tokens: {usage.output_tokens}")
print(f"📊 Total Tokens Used: {usage.total_tokens}")

------------------------------
📥 Input (Prompt) Tokens: 7198
📤 Output (Completion) Tokens: 580
📊 Total Tokens Used: 7778


## Chunking Long Documents

Full lesson pages can be thousands of characters, which makes retrieval less precise — a match deep inside a page still pulls in the whole page.

**Fix:** split each page into smaller, overlapping pieces using `chunk_documents`:
- `size=2000` — each chunk is a 2 000-character window
- `step=1000` — the window advances by 1 000 characters between chunks
- Overlap = `size - step = 1 000` characters, so no passage is split across a boundary

Every chunk keeps the original `filename` and adds `start` (byte offset) and `content` (chunk text).

In [24]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [25]:
# type(chunks)
chunks[:2]

[{'start': 0,
  'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour ph

### **Q4 — How many chunks do we get?**

In [26]:
# Q4: Count the total number of chunks produced
len(chunks)  # expected: ~295

295

In [27]:
# Build a separate minsearch index over the chunks
# (same field schema as the full-document index)

chunk_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
chunk_index.fit(chunks)

In [28]:
chunk_index.search("How does the agentic loop keep calling the model until it stops?")

[{'start': 4000,
  'content': 'while` loop. The loop keeps calling the model until\nit returns a response without any function calls. We also keep an\niteration counter so we can see how many round-trips happened.\n\n```python\nit = 1\n\nwhile True:\n    print(f"iteration #{it}...")\n    has_function_calls = False\n\n    response = openai_client.responses.create(\n        model="gpt-5.4-mini",\n        input=messages,\n        tools=[search_tool],\n    )\n\n    messages.extend(response.output)\n\n    for item in response.output:\n        if item.type == "function_call":\n            print("function_call:", item.name, item.arguments)\n            call_output = make_call(item)\n            messages.append(call_output)\n            has_function_calls = True\n\n        elif item.type == "message":\n            print("ASSISTANT:")\n            print(item.content[0].text)\n\n    it = it + 1\n    if has_function_calls == False:\n        break\n```\n\nThis is the core agent loop. The model rea

In [29]:
query="How does the agentic loop keep calling the model until it stops?"
search_results = search(query,index=chunk_index)
prompt = build_prompt(query, search_results)

input_messages = [
    {'role': 'developer', 'content': instructions},
    {'role': 'user', 'content': prompt}
]

response = llm_client.responses.create(
    model=model,
    input=input_messages
)

In [30]:
print(f"🤖 Response: {response.output_text}")

🤖 Response: The loop is a simple **`while True`** that repeats the whole request‑response cycle and only exits when the model gives a turn that contains **no function calls**.

1. **Start an iteration counter** (`it = 1`) and set a flag `has_function_calls = False`.  
2. **Call the model** (`openai_client.responses.create`) with the current `messages` list and the available tools (e.g., `search_tool`).  
3. **Append the model’s output** to the message history.  
4. **Inspect each item** in the output:  
   * If the item’s type is `"function_call"` the code runs the corresponding tool (`make_call(item)`), appends the tool’s result back to the message list, and flips `has_function_calls` to **True**.  
   * If the item is a normal `"message"` it just prints the assistant’s reply.  
5. Increment the iteration counter.  
6. **Exit condition:** after processing the whole response, the loop checks the flag.  
   ```python
   if has_function_calls == False:
       break
   ```  
   If the mod

In [31]:
# Token usage for the chunked-index RAG call (used in Q5 comparison)
usage = response.usage
print("-" * 30)
print(f"📥 Input (Prompt) Tokens: {usage.input_tokens}")
print(f"📤 Output (Completion) Tokens: {usage.output_tokens}")
print(f"📊 Total Tokens Used: {usage.total_tokens}")

------------------------------
📥 Input (Prompt) Tokens: 2381
📤 Output (Completion) Tokens: 491
📊 Total Tokens Used: 2872


### Repeating Q3 & Q5 using `RAGBase` 

#### Notes on `rag_helper.py` modifications -

The original `rag_helper.py` was written for the course FAQ schema (`section`/`question`/`answer`). To work with the lesson schema, the following adjustments were made:

**Schema adjustments (`RAGBase`)**
- `search` — removed `boost_dict` and `filter_dict` (no `course` field in lesson documents)
- `build_context` — updated to format `filename` + `content` pairs instead of `section`/`question`/`answer`

**Exposing token usage**
- `llm` — modified to return `(output_text, usage)` instead of just the response text
- `rag` — updated to unpack and return `(answer, usage)` as a tuple, so `usage.input_tokens` is accessible from outside the class

**Instructions**
- `INSTRUCTIONS` — updated to reflect that answers are drawn from course lesson content rather than FAQ entries

--- 
Two RAG instances are created:
- `rag1` — backed by the **full-document** index (one entry per lesson page)
- `rag2` — backed by the **chunk** index (2 000-char windows)

In [32]:
# long documents
index = Index(
    text_fields =["content"],
    keyword_fields=["filename"]
)
index.fit(documents)

# chunk documents
chunk_index = Index(
    text_fields =["content"],
    keyword_fields=["filename"]
)
chunk_index.fit(chunks)


In [33]:
from rag_helper import RAGBase

llm_client = get_groq_client()
model = "openai/gpt-oss-120b" 

rag1 = RAGBase(index=index,llm_client = llm_client,model = model)
rag2 = RAGBase(index=chunk_index,llm_client = llm_client,model = model)

query="How does the agentic loop keep calling the model until it stops?"

In [34]:
ans1,token_usage1 = rag1.rag(query)

In [35]:
print(f"🤖 Response 1: {ans1}")

🤖 Response 1: The loop is just a simple `while True` that repeats the request‑response cycle and checks whether the model asked for a tool call.

1. **Start a new iteration** – an iteration counter (`it`) is printed, and a flag `has_function_calls` is set to `False`.

2. **Call the model** with the current `messages` list (the full conversation history) and the list of tools.

3. **Process the response**:  
   * For each item in `response.output`  
     * If the item’s `type` is **`function_call`**, the code runs the requested tool (via `make_call`), appends the tool’s output back into `messages`, and sets `has_function_calls = True`.  
     * If the item’s `type` is **`message`**, it prints the assistant’s reply.

4. **Decide whether to continue** – after the for‑loop the code does:

```python
it = it + 1
if has_function_calls == False:
    break
```

If any function call occurred (`has_function_calls` is `True`), the loop goes back to step 1, sending the updated message history (whic

In [36]:
usage = token_usage1
print("-" * 30)
print(f"📥 Input (Prompt) Tokens: {usage.input_tokens}")
print(f"📤 Output (Completion) Tokens: {usage.output_tokens}")
print(f"📊 Total Tokens Used: {usage.total_tokens}")

------------------------------
📥 Input (Prompt) Tokens: 7198
📤 Output (Completion) Tokens: 465
📊 Total Tokens Used: 7663


In [37]:
ans2,token_usage2 = rag2.rag(query)

In [38]:
print(f"🤖 Response 2: {ans2}")

🤖 Response 2: The loop is just a **`while True`** that repeats the request‑response cycle and watches the model’s output for tool (function) calls.

1. **Start a turn** – set `has_function_calls = False`.  
2. **Call the model** (`openai_client.responses.create`) with the current message history (`messages`) and the list of available tools.  
3. **Inspect the response** – iterate over `response.output`.  
   * If an item’s `type` is **`function_call`**, the loop:
     * runs the corresponding tool (`make_call(item)`),
     * appends the tool’s result back into `messages`,
     * and sets `has_function_calls = True`.
   * If the item is a normal **`message`**, it records the assistant’s answer.  
4. **Advance the iteration counter** (optional, just for logging).  
5. **Exit check** – after processing all items, the loop does:

```python
if has_function_calls == False:
    break
```

If the model produced **no function calls** in that turn, the flag stays `False` and the loop breaks, ret

In [39]:
usage2 = token_usage2
print("-" * 30)
print(f"📥 Input (Prompt) Tokens: {usage2.input_tokens}")
print(f"📤 Output (Completion) Tokens: {usage2.output_tokens}")
print(f"📊 Total Tokens Used: {usage2.total_tokens}")

------------------------------
📥 Input (Prompt) Tokens: 2381
📤 Output (Completion) Tokens: 424
📊 Total Tokens Used: 2805


### **Q5 — How many fewer input tokens does the chunked version send?**

Divide the full-document input tokens by the chunk-index input tokens to find the reduction factor.

In [40]:
# Q5: Token reduction ratio — full-document RAG vs. chunked RAG
ratio = usage.input_tokens / usage2.input_tokens
print(f"Full-doc input tokens : {usage.input_tokens}")
print(f"Chunk input tokens    : {usage2.input_tokens}")
print(f"Ratio (full / chunk)  : {ratio:.2f}×")
# ~3× fewer tokens → answer: '3× fewer'

Full-doc input tokens : 7198
Chunk input tokens    : 2381
Ratio (full / chunk)  : 3.02×


## Agentic RAG

Now lets make the RAG *'agentic'* - instead of a fixed search-once pipeline, the LLM is given a `search` tool and left to decide *when* and *how many times* to call it.

The data is re-loaded here so this section can run independently without relying on the previous kernel session. The chunk index is rebuilt identically to Q4/Q5.

In [41]:
from gitsource import GithubRepositoryDataReader
from gitsource import chunk_documents

# long documents
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()
documents = []
for file in files:
    doc=file.parse()
    documents.append(doc)

 
# chunked documents
chunks = chunk_documents(documents, size=2000, step=1000)   

In [42]:
from minsearch import Index

# long docs index
index = Index(
    text_fields =["content"],
    keyword_fields=["filename"]
)
index.fit(documents)

# chunked docs index
chunk_index = Index(
    text_fields =["content"],
    keyword_fields=["filename"]
)
chunk_index.fit(chunks) 

In [43]:
# ToyAIKit — lightweight agent framework used in the module
# It reads typed Python functions + docstrings to build OpenAI tool schemas automatically

from toyaikit.llm import OpenAIChatCompletionsClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import (
    OpenAIChatCompletionsRunner, 
    DisplayingRunnerCallback,
    PricingConfig 
)

**Cost Tracking with ToyAIKit**

ToyAIKit supports cost simulation by registering a model's token pricing. Even when using a free-tier plan, registering the paid rates gives a realistic estimate of what the agentic loop would cost at scale.

This homework runs on the **`Groq free tier`**, which is rate-limited but sufficient for coursework — see [rate limits](https://console.groq.com/docs/rate-limits) and [paid pricing](https://groq.com/pricing).

**OpenRouter** offers the same model under a free tier as well — [free model](https://openrouter.ai/openai/gpt-oss-120b:free) · [paid pricing](https://openrouter.ai/openai/gpt-oss-120b).

In [44]:
# The pricing config below registers Groq's paid on-demand rates for cost simulation purposes.
pricing = PricingConfig()
pricing.register_model(
    "openai/gpt-oss-120b",  # model name
    0.15,                   # input cost per 1M tokens (USD)  — Groq on-demand rate
    0.60                    # output cost per 1M tokens (USD) — Groq on-demand rate
)

In [45]:
help(PricingConfig)

Help on class PricingConfig in module toyaikit.pricing:

class PricingConfig(builtins.object)
 |  Methods defined here:
 |
 |  __init__(self)
 |      Initialize self.  See help(type(self)) for accurate signature.
 |
 |  all_available_models(self)
 |      Lists all available models which has price data.
 |
 |      :return dict: Dictionary with provider as key and list of models as value
 |
 |  calculate_cost(self, model: str, input_tokens: int, output_tokens: int)
 |      Calculate cost for a LLM API call based on token usage.
 |
 |      Falls back to user-registered pricing when the model is not known to
 |      ``genai_prices``. Emits :class:`UnknownModelWarning` and returns ``None``
 |      when no pricing is available.
 |
 |      :param str model: Name of LLM model
 |      :param int input_tokens: Number of input tokens
 |      :param int output_tokens: Number of output tokens
 |      :return CostInfo | None: Object containing input cost, ouput cost and total cost, or None if model 

In [46]:
# Register `search` as an agent tool.
# ToyAIKit inspects the function's type hint and docstring to auto-generate the JSON schema.
# Using the framework to auto create and register the tool for the given function
# Here use Full-document index.Change`index`->`chunk_index` to use the chunked documents.
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"content": 3.0, "filename": 0.5},
        # filter_dict={"course": "llm-zoomcamp"}
    )

agent_tools = Tools()
agent_tools.add_tool(search)
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [47]:
from rag_helper import INSTRUCTIONS
INSTRUCTIONS

'\nYour task is to answer question from the lesson content based on the provided context.\n\nUse the context to find relevant information and provide accurate\nanswers. If the answer is not found in the lessons,\nrespond with "I don\'t know."\n'

In [50]:
# LLM client — wraps the Groq endpoint behind the OpenAI Chat Completions interface
llm_client = OpenAIChatCompletionsClient(
    model="openai/gpt-oss-120b",
    client=get_groq_client()
)

# Chat interface and callback for rendering the agent's output in the notebook
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

# Runner — wires together the tools, system prompt, LLM client, and pricing config
runner = OpenAIChatCompletionsRunner(
    tools=agent_tools,
    developer_prompt=INSTRUCTIONS,
    chat_interface=chat_interface,
    llm_client=llm_client,
    pricing_config=pricing 
)

# Kick off the agentic loop with the homework question
result = runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?",
    callback=callback,
)

-> Response received


-> Response received


Plain RAG (the classic pipeline),Agentic loop
"Fixed three‑step flow: search → build_prompt → LLM (one search, one model call)",A while‑loop that repeatedly sends the full conversation to the LLM and runs tools whenever the model requests them
No feedback from the model to the search step; if the first search is poor the answer is poor,"The model can inspect the search result, decide it’s insufficient, and issue another search (or a corrected query) automatically"
"No built‑in recovery from typos, ambiguous questions, or missing information","The model can repair typos, re‑phrase queries, ask clarifying questions, or combine multiple searches before answering"
Memory is limited to the single prompt you constructed; the LLM never sees the raw search results as separate messages,"The full message history (user prompt, developer instructions, each tool call, each tool result, each assistant message) is sent on every turn, giving the LLM context for iterative reasoning"
"Simpler, lower latency, cheaper (one API call)",More expensive and higher latency (potentially many API calls) but far more flexible and robust for complex or ambiguous queries


In [51]:
# Cost Info
cost_info = result.cost
token_info =result.tokens

# Convert decimal objects cleanly to floats
in_cost = float(cost_info.input_cost)
out_cost = float(cost_info.output_cost)
tot_cost = float(cost_info.total_cost)

print("📊 [SESSION RUNNER COST METRICS]")
print("=" * 40)
print(f"Model                       : {token_info.model}")
print(f"Input (Prompt) Tokens       : {token_info.input_tokens}")
print(f"Output (Completion) Tokens  : {token_info.output_tokens}")
print(f"📥 Input Token Cost  : ${in_cost:.8f}")
print(f"📤 Output Token Cost : ${out_cost:.8f}")
print("-" * 40)
print(f"💵 Total Session Cost: ${tot_cost:.8f}")
print("=" * 40)


📊 [SESSION RUNNER COST METRICS]
Model                       : openai/gpt-oss-120b
Input (Prompt) Tokens       : 7121
Output (Completion) Tokens  : 1100
📥 Input Token Cost  : $0.00106815
📤 Output Token Cost : $0.00066000
----------------------------------------
💵 Total Session Cost: $0.00172815


### **Q6 — How many times did the agent call `search`?**

`result.all_messages` contains one entry per conversation turn (user message, each tool call + result, and the final assistant reply).

With 4 `search` calls and 1 final answer turn, the runner records **5 messages** in total.

In [52]:
help(result)

Help on LoopResult in module toyaikit.chat.runners object:

class LoopResult(typing.Generic)
 |  LoopResult(
 |      new_messages: list,
 |      all_messages: list,
 |      tokens: toyaikit.pricing.TokenUsage,
 |      cost: toyaikit.pricing.CostInfo | None,
 |      last_message: ~T
 |  ) -> None
 |
 |  LoopResult(new_messages: list, all_messages: list, tokens: toyaikit.pricing.TokenUsage, cost: toyaikit.pricing.CostInfo | None, last_message: ~T)
 |
 |  Method resolution order:
 |      LoopResult
 |      typing.Generic
 |      builtins.object
 |
 |  Methods defined here:
 |
 |  __eq__(self, other)
 |      Return self==value.
 |
 |  __init__(
 |      self,
 |      new_messages: list,
 |      all_messages: list,
 |      tokens: toyaikit.pricing.TokenUsage,
 |      cost: toyaikit.pricing.CostInfo | None,
 |      last_message: ~T
 |  ) -> None
 |      Initialize self.  See help(type(self)) for accurate signature.
 |
 |  __replace__ = _replace(self, /, **changes) from dataclasses
 |
 |  __re

In [53]:
result.all_messages

[{'role': 'system',
  'content': '\nYour task is to answer question from the lesson content based on the provided context.\n\nUse the context to find relevant information and provide accurate\nanswers. If the answer is not found in the lessons,\nrespond with "I don\'t know."\n'},
 {'role': 'user',
  'content': 'How does the agentic loop work, and how is it different from plain RAG?'},
 ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='fc_a51fe9e5-473c-499c-984d-b26c08933b86', function=Function(arguments='{"query":"agentic loop plain RAG"}', name='search'), type='function')], reasoning='The user asks: "How does the agentic loop work, and how is it different from plain RAG?" We need to answer based on provided context from lesson content. We don\'t have the lesson content directly. We might need to search the FAQ database. Use function search.'),
 {'role': 'tool',
  '

In [54]:
# Q6: Count how many times the agent called search
tool_responses = sum(1 for msg in result.all_messages if isinstance(msg, dict) and msg.get('role') == 'tool')

print(f"Total messages in session : {len(result.all_messages)}")
print(f"Total search tool calls   : {tool_responses}")


Total messages in session : 5
Total search tool calls   : 1
